# Task 3 - RNN question-generation model

A from-scratch 2-layer bidirectional GRU encoder, 2-layer GRU decoder, and Luong attention. No pretrained models, embeddings, Transformers, or seq2seq libraries are used. Run the 10k-pair debug run first; when loss falls, set `DEBUG = False` for the full 10-15 epoch run.

In [2]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import sentencepiece as spm
import torch
import torch.nn as nn
from google.colab import drive
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset, Subset

PAD, BOS, EOS = 0, 2, 3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using:', DEVICE)

In [ ]:
# Task 1 and Task 2 artifacts, plus all Task 3 results, are saved on Drive.
drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/GenAI-Dataset')
MODEL_DIR = DRIVE_DIR / 'model'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_PATH = DRIVE_DIR / 'train.tsv'
VALID_PATH = DRIVE_DIR / 'valid.tsv'
SP_MODEL_PATH = DRIVE_DIR / 'ur_sp.model'
for path in [TRAIN_PATH, VALID_PATH, SP_MODEL_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'Missing prerequisite: {path}')
print('Task 3 outputs:', MODEL_DIR)


In [ ]:
class Seq2SeqDataset(Dataset):
    def __init__(self, tsv_path, sp_model_path):
        self.sp = spm.SentencePieceProcessor(model_file=str(sp_model_path))
        with open(tsv_path, encoding='utf-8') as file:
            reader = csv.reader(file, delimiter='\t', quoting=csv.QUOTE_NONE, escapechar='\\')
            self.pairs = [(row[0], row[1]) for row in reader if len(row) == 2]

    def __len__(self): return len(self.pairs)

    def __getitem__(self, index):
        source, target = self.pairs[index]
        source_ids = self.sp.encode(source, out_type=int)
        target_ids = [BOS] + self.sp.encode(target, out_type=int) + [EOS]
        return torch.tensor(source_ids), torch.tensor(target_ids)

def collate_fn(batch):
    sources, targets = zip(*batch)
    lengths = torch.tensor([len(x) for x in sources], dtype=torch.long)
    return (pad_sequence(sources, batch_first=True, padding_value=PAD), lengths,
            pad_sequence(targets, batch_first=True, padding_value=PAD))

train_dataset = Seq2SeqDataset(TRAIN_PATH, SP_MODEL_PATH)
valid_dataset = Seq2SeqDataset(VALID_PATH, SP_MODEL_PATH)
VOCAB_SIZE = train_dataset.sp.get_piece_size()
print(f'Train: {len(train_dataset):,}; valid: {len(valid_dataset):,}; vocab: {VOCAB_SIZE:,}')

## Model

The encoder uses `pack_padded_sequence` so padded source tokens do not affect GRU states. Luong attention compares each projected encoder state with the decoder state using batched matrix multiplication. The loss ignores target padding with `CrossEntropyLoss(ignore_index=PAD)`. See the relevant PyTorch documentation for [packing](https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_padded_sequence.html), [GRU](https://pytorch.org/docs/stable/generated/torch.nn.GRU.html), and [cross-entropy](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html).

In [ ]:
class LuongAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.project = nn.Linear(hidden_size * 2, hidden_size, bias=False)

    def forward(self, encoder_outputs, decoder_hidden, source_mask):
        scores = torch.bmm(self.project(encoder_outputs), decoder_hidden.unsqueeze(2)).squeeze(2)
        weights = torch.softmax(scores.masked_fill(~source_mask, -1e9), dim=1)
        return torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1), weights

class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_size=256, hidden_size=512, layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_size, padding_idx=PAD)
        self.dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(embedding_size, hidden_size, layers, batch_first=True,
                            bidirectional=True, dropout=dropout)
        self.hidden_bridge = nn.Linear(hidden_size * 2, hidden_size)
        self.layers = layers

    def forward(self, source, lengths):
        packed = pack_padded_sequence(self.dropout(self.embedding(source)), lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        packed_outputs, hidden = self.gru(packed)
        outputs, _ = pad_packed_sequence(packed_outputs, batch_first=True, total_length=source.size(1))
        hidden = hidden.view(self.layers, 2, source.size(0), -1)
        hidden = torch.tanh(self.hidden_bridge(torch.cat([hidden[:, 0], hidden[:, 1]], dim=-1)))
        return outputs, hidden

class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_size=256, hidden_size=512, layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_size, padding_idx=PAD)
        self.dropout = nn.Dropout(dropout)
        self.attention = LuongAttention(hidden_size)
        self.gru = nn.GRU(embedding_size + hidden_size * 2, hidden_size, layers,
                            batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_size * 3, vocab_size)

    def step(self, token, hidden, encoder_outputs, source_mask):
        context, weights = self.attention(encoder_outputs, hidden[-1], source_mask)
        embedded = self.dropout(self.embedding(token)).unsqueeze(1)
        output, hidden = self.gru(torch.cat([embedded, context.unsqueeze(1)], dim=-1), hidden)
        logits = self.output(torch.cat([output.squeeze(1), context], dim=-1))
        return logits, hidden, weights

class Seq2Seq(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.encoder = Encoder(vocab_size)
        self.decoder = Decoder(vocab_size)

    def forward(self, source, lengths, target):
        encoder_outputs, hidden = self.encoder(source, lengths)
        source_mask = source.ne(PAD)
        logits = []
        # Teacher forcing: input token t is the true target token at t.
        for time_step in range(target.size(1) - 1):
            step_logits, hidden, _ = self.decoder.step(target[:, time_step], hidden, encoder_outputs, source_mask)
            logits.append(step_logits)
        return torch.stack(logits, dim=1)

model = Seq2Seq(VOCAB_SIZE).to(DEVICE)
parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {parameter_count:,}')

In [ ]:
# Debug on 10k pairs. A falling loss after one epoch confirms the model is learning.
DEBUG = False
BATCH_SIZE, EPOCHS, LEARNING_RATE = 64, (2 if DEBUG else 10), 1e-3
PATIENCE = 2
if DEBUG:
    train_data = Subset(train_dataset, range(min(10_000, len(train_dataset))))
else: train_data = train_dataset
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    with torch.set_grad_enabled(training):
        for source, lengths, target in loader:
            source, lengths, target = source.to(DEVICE), lengths.to(DEVICE), target.to(DEVICE)
            logits = model(source, lengths, target)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), target[:, 1:].reshape(-1))
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
    return total_loss / len(loader)

history, best_valid_loss, epochs_without_improvement = [], float('inf'), 0
checkpoint_path = MODEL_DIR / 'best_model.pt'
for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, training=True)
    valid_loss = run_epoch(valid_loader, training=False)
    history.append({'epoch': epoch, 'train_loss': train_loss, 'valid_loss': valid_loss})
    print(f'Epoch {epoch:02d} | train: {train_loss:.4f} | valid: {valid_loss:.4f}')
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), checkpoint_path)
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f'Early stopping after {PATIENCE} epochs without improvement.')
            break

with open(MODEL_DIR / 'loss_history.csv', 'w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=['epoch', 'train_loss', 'valid_loss'])
    writer.writeheader(); writer.writerows(history)
print(f'Best valid loss: {best_valid_loss:.4f}; saved: {checkpoint_path}')

In [ ]:
# Required loss plot, also saved on Google Drive.
plt.figure(figsize=(7, 4))
plt.plot([x['epoch'] for x in history], [x['train_loss'] for x in history], marker='o', label='train')
plt.plot([x['epoch'] for x in history], [x['valid_loss'] for x in history], marker='o', label='validation')
plt.xlabel('Epoch'); plt.ylabel('Cross-entropy loss'); plt.legend(); plt.tight_layout()
plt.savefig(MODEL_DIR / 'loss_curve.png', dpi=160)
plt.show()

## Greedy and beam decoding

These functions load the best validation-loss checkpoint. Beam search uses beam size 3, within the required range of 3-5.

In [ ]:
model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
model.eval()
sp = train_dataset.sp

@torch.no_grad()
def encode_source(source_text):
    source_ids = torch.tensor([sp.encode(source_text, out_type=int)], device=DEVICE)
    lengths = torch.tensor([source_ids.size(1)], device=DEVICE)
    outputs, hidden = model.encoder(source_ids, lengths)
    return outputs, hidden, source_ids.ne(PAD)

@torch.no_grad()
def greedy_decode(source_text, max_length=30):
    outputs, hidden, mask = encode_source(source_text)
    token, generated = torch.tensor([BOS], device=DEVICE), []
    for _ in range(max_length):
        logits, hidden, _ = model.decoder.step(token, hidden, outputs, mask)
        token = logits.argmax(dim=-1)
        if token.item() == EOS: break
        generated.append(token.item())
    return sp.decode(generated)

@torch.no_grad()
def beam_decode(source_text, beam_size=3, max_length=30):
    outputs, hidden, mask = encode_source(source_text)
    beams = [([], 0.0, torch.tensor([BOS], device=DEVICE), hidden, False)]
    for _ in range(max_length):
        candidates = []
        for tokens, score, token, beam_hidden, ended in beams:
            if ended:
                candidates.append((tokens, score, token, beam_hidden, True)); continue
            logits, next_hidden, _ = model.decoder.step(token, beam_hidden, outputs, mask)
            values, ids = torch.log_softmax(logits, dim=-1).topk(beam_size, dim=-1)
            for value, token_id in zip(values[0], ids[0]):
                next_id = token_id.item()
                candidates.append((tokens + [next_id], score + value.item(), token_id.unsqueeze(0),
                    next_hidden.clone(), next_id == EOS))
        beams = sorted(candidates, key=lambda b: b[1] / max(1, len(b[0])), reverse=True)[:beam_size]
        if all(beam[4] for beam in beams): break
    best = beams[0][0]
    return sp.decode(best[:best.index(EOS)] if EOS in best else best)

source, reference = valid_dataset.pairs[0]
print('Source:   ', source)
print('Reference:', reference)
print('Greedy:   ', greedy_decode(source))
print('Beam:     ', beam_decode(source, beam_size=3))